# 00 — Setup and preprocessing

Run this **once per Drive**. Cells 1–4 also run at the start of every session,
because Colab discards the compiled extensions; cells 5–8 are one-time.

Order matters: undistortion must precede the dense clouds, because
`roma_init` uses the same scene reader.


## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/SeathruNeRF_dataset/   original, as downloaded
#     dataset/undistorted/<scene>/   PINHOLE + sparse/0/  <- required
#     dense/<scene>.ply|.json        M1 clouds, SHA-256 sidecars
#     run_ledger.json                campaign state
#     runs/<cell>/<scene>/s<seed>/   one run, all of it together
#     analysis/                      analyse.py output, figures, tables
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATA_ORIG    = f'{DRIVE_ROOT}/dataset/SeathruNeRF_dataset'
DATA_UNDIST  = f'{DRIVE_ROOT}/dataset/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
for d in (DRIVE_ROOT, DATA_UNDIST, DENSE_DIR, ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)
print('drive root:', DRIVE_ROOT)


## 2. GPU — must be an A100

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


## 3. Clone and build

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout)

# Builds diff_gaussian_rasterization_ms and simple_knn for sm_80, and installs
# only the dependencies Colab does not already ship.
!bash tools/setup_colab.sh


## 4. Verify the rasterizer merge

In [ ]:
# The gate. The whole rasterizer merge rests on one identity: for a single
# Gaussian at depth z the probe gives Z_raw = alpha*z, so Z_raw/alpha must
# recover z on every covered pixel. Verified on sm_86 during development; this
# confirms it on sm_80 before anything is trained on top of it.
%cd $IMPL_DIR
!python -m tools.verify_rasterizer


## 5. The rest of the self-checks

Sixty-nine checks across nine suites. Cheap, and several encode findings that
are easy to reintroduce.

In [ ]:
for t in ['verify_config_layer','verify_ledger','verify_metrics','verify_storage',
          'verify_analysis','verify_undistort','verify_dense_init','verify_simplify',
          'verify_quantize']:
    !python -m tools.{t} 2>&1 | tail -2


## 6. Dataset → Drive

Place `SeathruNeRF_dataset/` under `dataset/` on Drive once. The four scenes
are Curasao (21 images), IUI3-RedSea (29, note the capital-I `Images_wb`),
JapaneseGradens-RedSea (20) and Panama (18).

In [ ]:
import os
missing = [s for s in SCENES if not os.path.exists(f'{DATA_ORIG}/{s}')]
assert not missing, (
    f'Missing scenes under {DATA_ORIG}: {missing}\n'
    f'Upload SeathruNeRF_dataset there first.')
for s in SCENES:
    d = [x for x in os.listdir(f'{DATA_ORIG}/{s}') if x.lower()=='images_wb'][0]
    n = len(os.listdir(f'{DATA_ORIG}/{s}/{d}'))
    print(f'{s:24s} {n:3d} images   image dir: {d}')


## 7. COLMAP undistortion — **required**

All four scenes ship with the COLMAP **OPENCV** camera model and real
distortion coefficients, while the scene reader accepts only
PINHOLE/SIMPLE_PINHOLE. Without this step every run fails at scene load.

The step is idempotent, so re-running this notebook is harmless.

In [ ]:
!apt-get -qq install colmap > /dev/null 2>&1 || pip install -q pycolmap
import subprocess
for s in SCENES:
    print(f'--- {s} ---')
    r = subprocess.run(['python','-m','source.undistort',
                        '--source', f'{DATA_ORIG}/{s}',
                        '--output', f'{DATA_UNDIST}/{s}'],
                       capture_output=True, text=True)
    print(r.stdout[-800:] or r.stderr[-800:])


In [ ]:
# Confirm every scene will now load.
from pathlib import Path
import sys; sys.path.insert(0, IMPL_DIR)
from source.undistort import verify_undistorted
for s in SCENES:
    info = verify_undistorted(Path(DATA_UNDIST)/s)
    print(f'{s:24s} {info["model"]:16s} {info["width"]}x{info["height"]}')


## 8. Dense clouds — for the M1 cells (A1, A4, A5, A7)

One per scene. The preset is a real experimental choice: with densification
disabled the primitive count can never grow, so a cloud below the budget makes
A4 collapse onto A1 and A7 onto A5. Check the reported point counts against the
budget once S1 has produced one.

Preprocessing wall-clock is **not** part of training time — report it
alongside, or A1's cost is understated relative to A0's.

In [ ]:
import subprocess
for s in SCENES:
    print(f'--- {s} ---')
    r = subprocess.run(['python','-m','source.roma_init',
                        '--source_path', f'{DATA_UNDIST}/{s}',
                        '--output',      f'{DENSE_DIR}/{s}.ply',
                        '--images',      'images',
                        '--preset',      'sparse',
                        '--seed',        '0'],
                       capture_output=True, text=True)
    print(r.stdout[-900:] or r.stderr[-900:])


## 9. Initialise the ledger

96 rows: 8 cells × 4 scenes × 3 seeds. Refuses to overwrite an existing
campaign unless forced.

In [ ]:
!python -m tools.run_ledger init --output_root "$DRIVE_ROOT"
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


---
Next: open **01_worker.ipynb** and run it. Repeat it every session until the
ledger reports everything done.
